In [ ]:
import os
import gc
import json
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DATASET_PATH = "/kaggle/input/dataset-combined/dataset_combined.csv"

In [ ]:
# LOAD DATASET
df = pd.read_csv(DATASET_PATH)

print("Dataset shape:", df.shape)
print(df.head())

# Combine title + abstract
df["text"] = df["title"].astype(str) + "\n\n" + df["abstract"].astype(str)

In [ ]:
# LABEL ENCODING

# L1 Labels (7 Main Categories)
main_labels = sorted(df["main_label"].unique())

main_label2id = {
    label: idx
    for idx, label in enumerate(main_labels)
}

main_id2label = {
    idx: label
    for label, idx in main_label2id.items()
}

df["main_label_id"] = df["main_label"].map(main_label2id)

print("\nL1 Main Categories:")
print(main_label2id)

# L2 Labels (42 Subcategories)

sub_labels = sorted(df["sub_label"].unique())

sub_label2id = {
    label: idx
    for idx, label in enumerate(sub_labels)
}

sub_id2label = {
    idx: label
    for label, idx in sub_label2id.items()
}

df["sub_label_id"] = df["sub_label"].map(sub_label2id)

print("\nL2 Subcategories:")
print(sub_label2id)

# Save mappings

os.makedirs("label_mappings", exist_ok=True)

with open("label_mappings/main_label2id.json", "w") as f:
    json.dump(main_label2id, f, indent=2)

with open("label_mappings/main_id2label.json", "w") as f:
    json.dump(main_id2label, f, indent=2)

with open("label_mappings/sub_label2id.json", "w") as f:
    json.dump(sub_label2id, f, indent=2)

with open("label_mappings/sub_id2label.json", "w") as f:
    json.dump(sub_id2label, f, indent=2)

In [ ]:
# TRAIN / VAL / TEST SPLIT
# Stratify by sub_label only
# This automatically balances main categories too
# because every subcategory belongs to exactly one main category

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["sub_label"],
    random_state=SEED
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["sub_label"],
    random_state=SEED
)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

# Save splits for reproducibility
train_df.to_csv("train_split.csv", index=False)
val_df.to_csv("val_split.csv", index=False)
test_df.to_csv("test_split.csv", index=False)

In [ ]:
# TOKENIZER
MODEL_NAME = "allenai/scibert_scivocab_uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# L1 Tokenization
# Abstract only
def tokenize_l1(batch):

    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=512
    )
    
# L2 Tokenization
# Prepends main category hint
def tokenize_l2(batch):

    hints = [
        f"Main category: {main}\n\n{text}"
        for main, text in zip(batch["main_label"], batch["text"])
    ]

    return tokenizer(
        hints,
        truncation=True,
        max_length=512
    )

In [ ]:
# CONVERT TO HUGGINGFACE DATASETS

# L1 Datasets
train_l1 = Dataset.from_pandas(
    train_df[["text", "main_label_id"]]
)

val_l1 = Dataset.from_pandas(
    val_df[["text", "main_label_id"]]
)

# Rename label column
train_l1 = train_l1.rename_column("main_label_id", "labels")
val_l1 = val_l1.rename_column("main_label_id", "labels")

# Tokenize
train_l1 = train_l1.map(
    tokenize_l1,
    batched=True
)

val_l1 = val_l1.map(
    tokenize_l1,
    batched=True
)

# Remove unused columns
train_l1 = train_l1.remove_columns(["text"])
val_l1 = val_l1.remove_columns(["text"])

# PyTorch format
train_l1.set_format("torch")
val_l1.set_format("torch")

In [ ]:
# LOAD L1 MODEL
model_l1 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(main_labels),
    id2label=main_id2label,
    label2id=main_label2id
)

In [ ]:
# METRICS
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    weighted_f1 = f1_score(
        labels,
        predictions,
        average="weighted"
    )

    return {
        "accuracy": accuracy,
        "weighted_f1": weighted_f1
    }


In [ ]:
# TRAINING ARGUMENTS (L1)

training_args_l1 = TrainingArguments(

    output_dir="./scibert_l1",

    evaluation_strategy="epoch",
    save_strategy="epoch",

    save_total_limit=1,

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    num_train_epochs=3,

    weight_decay=0.01,

    fp16=True,

    gradient_checkpointing=True,

    logging_steps=100,

    load_best_model_at_end=True,

    metric_for_best_model="weighted_f1",

    greater_is_better=True,

    report_to="none"
)


In [ ]:
# TRAINER (L1)
trainer_l1 = Trainer(

    model=model_l1,

    args=training_args_l1,

    train_dataset=train_l1,

    eval_dataset=val_l1,

    tokenizer=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

# train l1
trainer_l1.train()

In [ ]:
# save L1
trainer_l1.save_model("./scibert_l1")
tokenizer.save_pretrained("./scibert_l1")

print("L1 model saved.")

In [ ]:
# FREE MEMORY BEFORE L2

del train_l1
del val_l1
del trainer_l1
del model_l1

gc.collect()
torch.cuda.empty_cache()

print("L1 memory cleared.")

In [ ]:
# BUILD L2 DATASETS

train_l2 = Dataset.from_pandas(
    train_df[["text", "main_label", "sub_label_id"]]
)

val_l2 = Dataset.from_pandas(
    val_df[["text", "main_label", "sub_label_id"]]
)

# Rename label column
train_l2 = train_l2.rename_column("sub_label_id", "labels")
val_l2 = val_l2.rename_column("sub_label_id", "labels")

# Tokenize
train_l2 = train_l2.map(
    tokenize_l2,
    batched=True
)

val_l2 = val_l2.map(
    tokenize_l2,
    batched=True
)

# Remove unused columns
train_l2 = train_l2.remove_columns(["text", "main_label"])
val_l2 = val_l2.remove_columns(["text", "main_label"])

# Torch format
train_l2.set_format("torch")
val_l2.set_format("torch")

In [ ]:
# LOAD L2 MODEL
model_l2 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(sub_labels),
    id2label=sub_id2label,
    label2id=sub_label2id
)

In [ ]:
# TRAINING ARGUMENTS (L2)
training_args_l2 = TrainingArguments(

    output_dir="./scibert_l2",

    evaluation_strategy="epoch",
    save_strategy="epoch",

    save_total_limit=1,

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    num_train_epochs=3,

    weight_decay=0.01,

    fp16=True,

    gradient_checkpointing=True,

    logging_steps=100,

    load_best_model_at_end=True,

    metric_for_best_model="weighted_f1",

    greater_is_better=True,

    report_to="none"
)

In [ ]:
# TRAINER (L2)
trainer_l2 = Trainer(

    model=model_l2,

    args=training_args_l2,

    train_dataset=train_l2,

    eval_dataset=val_l2,

    tokenizer=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

# train L2
trainer_l2.train()

In [ ]:
# save L2
trainer_l2.save_model("./scibert_l2")
tokenizer.save_pretrained("./scibert_l2")

print("L2 model saved.")

# SAVE TEST SET
test_df.to_csv("test_set.csv", index=False)

print("Test set saved.")

In [ ]:
# FINAL NOTES
print("""
============================================================

TRAINING COMPLETE

Download these folders/files from Kaggle:

1. scibert_l1/
2. scibert_l2/
3. test_set.csv
4. label_mappings/

Place models into:

backend/models/scibert_l1/
backend/models/scibert_l2/

The REAL hierarchical evaluation
(L1 -> top-K -> L2 masking -> combined score)
will be implemented in classifier.py.

============================================================
""")